In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import glob
import json

In [2]:
def processFile(file_path, volume_number=None):
    with open(file_path, 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file, 'html.parser')
    h3_text = soup.find('h3').text.strip().split(':', 1)
    parva_name=h3_text[0].strip()
    if len(parva_name)<=3:
        print("Parva name not found.", file_path)
    chapter_number= int(re.search(r'Chapter\s+(\d+)', h3_text[1].strip()).group(1))
    if not chapter_number:
        print("Chapter number not found.", file_path)
    if soup.find('h4'):
        chapter_title = soup.find('h4').text.strip()
    else:
        print("Chapter title not found.", file_path)

    sentences = []
    p_tags = soup.find_all('p', id=True)

    for p_tag in p_tags:
        sentence_id = p_tag.get('id', '').strip()
        text_content = str(p_tag) # Get the text content and split by <br />
        text_content = re.sub(r'<a[^>]*>.*?</a>', '', text_content) # Remove the <a> tag content: Sanskrit Text <br> Translation
        p_soup = BeautifulSoup(text_content, 'html.parser') # Parse with BeautifulSoup to handle the <br /> tag
        br_tag = p_soup.find('br')
        if br_tag:
            parts = str(p_soup).split('<br/>') # Split at <br /> - Sanskrit before, translation after
            if len(parts) > 1:
                translated_text = BeautifulSoup(parts[1], 'html.parser').get_text().strip() # Extract just the translated text (after <br />)
                translated_text = re.sub(r'<[^>]+>', '', translated_text).strip() # Clean up any remaining HTML tags
                sentences.append({'sentence_id': sentence_id,'translated_text': translated_text})
    df = pd.DataFrame(sentences)
    df['parva_name'] = parva_name
    df['chapter_number'] = chapter_number
    df['chapter_title'] = chapter_title
    df['file_name'] = os.path.basename(file_path)
    df['volume_number'] = volume_number
    df = df[['parva_name', 'chapter_number', 'chapter_title', 'sentence_id', 'translated_text', 'file_name', 'volume_number']]
    return df

In [3]:
directory_path = "./mahabharata/itihasa/gen/mahabharata"
volumes = [item for item in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, item))]
df_list = []
i=0
for v in volumes:
    for f in glob.glob(os.path.join(directory_path, v, "*.html"), recursive=False):
        if os.path.basename(f) != "index.html":
            i+=1
            df_list.append(processFile(f, volume_number=v))
print("Total files processed:", i)
df = pd.concat(df_list, ignore_index=True)
print("Parsed DataFrame shape:", df.shape)
df.head()

Total files processed: 2110
Parsed DataFrame shape: (73659, 7)


,parva_name,chapter_number,chapter_title,sentence_id,translated_text,file_name,volume_number
0,ANUKRAMANIKA PARVA,1,Contents of subject,1,Having saluted the Supreme Deity (Narayana) an...,1.html,vol-i
1,ANUKRAMANIKA PARVA,1,Contents of subject,2,One day when the great sages of hard austeriti...,1.html,vol-i
2,ANUKRAMANIKA PARVA,1,Contents of subject,3,(Thereupon) desirous of hearing his wonderful ...,1.html,vol-i
3,ANUKRAMANIKA PARVA,1,Contents of subject,4,Having been welcomed with due respect by those...,1.html,vol-i
4,ANUKRAMANIKA PARVA,1,Contents of subject,5,"After the Rishis had taken their seats, Lomaha...",1.html,vol-i


In [4]:
# !pip install -U transformers==4.40.2

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from transformers import LlamaTokenizer
from typing import List, Dict, Any
import re
import json

In [6]:
class MahabharataAnnotator:
    def __init__(self, model_name="microsoft/phi-2"):
        """Initialize the LLM annotator for Phi-2 model"""
        print(f"Loading model: {model_name}")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None, trust_remote_code=True)
        
        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=False
        )
        
        # Entity and relation definitions
        self.entity_types = {
            "CHARACTER": ["PERSON", "CHARACTER_GROUP", "DIVINE_BEING", "MYTHICAL_BEING"],
            "LOCATION": ["CITY", "KINGDOM", "BATTLEFIELD", "FOREST", "RIVER", "MOUNTAIN", "HEAVENLY_REALM"],
            "EVENT": ["BATTLE_EVENT", "RITUAL", "COURT_EVENT", "EXILE", "DIVINE_EVENT", "POLITICAL_EVENT"],
            "OBJECT": ["WEAPON", "JEWELRY", "VEHICLE", "TEXT", "TREASURE"],
            "CONCEPT": ["DHARMIC_CONCEPT", "VIRTUE", "VICE", "EMOTION", "PHILOSOPHICAL"],
            "TIME": ["YUGAS", "ERA", "SEASON", "FESTIVAL"]
        }
        self.relation_types = {
            "FAMILY RELATIONS": ["PARENT_OF","CHILD_OF","SPOUSE_OF","SIBLING_OF","COUSIN_OF","GRANDPARENT_OF","GRANDCHILD_OF","IN_LAW_OF"],
            "SOCIAL RELATIONS": ["FRIEND_OF","ALLY_OF","ENEMY_OF","RIVAL_OF","TEACHER_OF","STUDENT_OF","MASTER_OF","SERVANT_OF","PROTECTOR_OF","PROTECTED_BY","COMPANION_OF"],
            "ACTION RELATIONS": ["KILLED_BY","KILLER_OF","INJURED_BY","INJURER_OF","ATTACKED_BY","ATTACKER_OF","DEFEATED_BY","DEFEATER_OF","SPARED_BY","BETRAYED_BY","BETRAYER_OF","FORGAVE","FORGIVEN_BY","CURSED_BY","CURSER_OF","BLESSED_BY","BLESSER_OF"],
            "ROLE RELATIONS": ["RULER_OF","RULED_BY","COMMANDER_OF","COMMANDED_BY","CHARIOEER_OF","ADVISOR_TO","ADVISED_BY","REPRESENTATIVE_OF","INCARNATION_OF"],
            "SPATIAL RELATIONS": ["LOCATED_AT","ORIGIN_FROM","TRAVELED_TO","TRAVELED_FROM","EXILED_TO","RETURNED_TO"]
        }
        print("Model loaded successfully!")
    
    def create_simplified_prompt(self, text: str) -> str:
        prompt = f"""You are analyzing Mahabharata text. Extract entities and relationships.
        Entity types:"{list(self.entity_types.keys())}"
        Relationship types: "{list(self.relation_types.keys())}"

        Instructions:
        1. List ALL entities with exact text and type
        2. List ALL relationships between entities
        3. ONLY Return as JSON with 'entities' and 'relations' arrays

        Example format:
        {{
        "entities": [
            {{"text": "Krishna", "type": "CHARACTER", "start": 0, "end": 7}},
            {{"text": "Brahma", "type": "CHARACTER", "start": 7, "end": 13}} ...
        ],
        "relations": [
            {{"entity1": "Krishna", "relation": "TEACHER_OF", "entity2": "Arjuna"}}
            {{"entity1": "Krishna", "relation": "IN_LAW_OF", "entity2": "Arjuna"}} ...
        ]
        }}

        Now analyze this text: "{text}"

        Response:"""
        return prompt
    
    def extract_annotations(self, text: str) -> Dict[str, Any]:
        """Extract annotations using Phi-2"""
        
        prompt = self.create_simplified_prompt(text)
        response = self.generator(
            prompt, max_length=len(prompt) + 200, num_return_sequences=1,
            pad_token_id=self.tokenizer.eos_token_id)[0]['generated_text']
        response = response.replace(prompt, "").strip()
        self.response = response
        validated = {"entities": [], "relations": []}
        for match in re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', response, re.DOTALL):
            try:
                data=(match.strip().replace("'", '"').replace("None", "null").replace("True", "true").replace("False", "false"))
                data = json.loads(re.sub(r',\s*]', ']', re.sub(r',\s*}', '}', data)))
                if all(key in data for key in ["text", "type"]):
                    validated["entities"].append({"text": data["text"],"type": data["type"],"start": data.get("start", 0),"end": len(data["text"]) + data.get("start", 0)})
                elif ("entities" in data):
                    for entity in data["entities"]:
                        if all(key in entity for key in ["text", "type"]):
                            validated["entities"].append({"text": entity["text"],"type": entity["type"],"start": entity.get("start", 0),"end": len(entity["text"]) + entity.get("start", 0)})
                if all(key in data for key in ["entity1", "relation", "entity2"]):
                                validated["relations"].append({"entity1": data["entity1"],"relation": data["relation"],"entity2": data["entity2"]})
                elif ("relations" in data):
                    for relation in data["relations"]:
                        if all(key in relation for key in ["entity1", "relation", "entity2"]):
                            validated["relations"].append({"entity1": relation["entity1"],"relation": relation["relation"], "entity2": relation["entity2"]})
            except json.JSONDecodeError:
                continue
        return validated
    def batch_annotate(self, texts: List[str], batch_size: int = 2) -> List[Dict]:
        """Annotate multiple texts"""
        all_annotations = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            print(f"Processing batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")
            for text in batch:
                annotations = self.extract_annotations(text)
                all_annotations.append(annotations)
        return all_annotations

In [7]:
df = df
sample_size = min(1000, len(df))
df_sample = df.sample(n=sample_size).copy()
print("Initializing annotator...")
annotator = MahabharataAnnotator()
texts = df_sample['translated_text'].tolist()
annotations = annotator.batch_annotate(texts, batch_size=2)
print("Converting to doccano format...")

output_records = []
for idx, (text, annotation) in enumerate(zip(texts, annotations)):
    meta = {
        'unique_id': f"{df.loc[idx, 'parva_name']}_{df.loc[idx, 'chapter_number']}_{df.loc[idx, 'sentence_id']}",
        'parva_name': df.loc[idx, 'parva_name'],
        'chapter': str(df.loc[idx, 'chapter_number']),
        'sentence': str(df.loc[idx, 'sentence_id'])
    }
    entities = []
    for entity in annotation.get('entities', []):
        entities.append({
            "start_offset": entity.get("start", 0),
            "end_offset": entity.get("end", len(entity.get("text", ""))),
            "label": entity.get("type", "ENTITY")
        })
    output_records.append({
        "text": text,
        "meta": meta,
        "entities": entities,
        "relations": annotation.get('relations', [])
    })

output_path = "./mahabharata_preannotated.jsonl"
with open(output_path, 'w', encoding='utf-8') as f:
    for record in output_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
print(f"✅ Pre-annotation complete! Saved to {output_path}")
# Show sample
print("\nSample annotation:")
print(json.dumps(output_records[0], indent=2, ensure_ascii=False))


Initializing annotator...
Loading model: microsoft/phi-2


/opt/conda/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Both `max_new_tokens` (=300) and `max_length`(=1271) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded successfully!
Processing batch 1/500


...

Processing batch 499/500


Both `max_new_tokens` (=300) and `max_length`(=1417) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=1438) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing batch 500/500


Both `max_new_tokens` (=300) and `max_length`(=1285) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Converting to doccano format...
✅ Pre-annotation complete! Saved to ./mahabharata_preannotated.jsonl

Sample annotation:
{
  "text": "All the combatants also, and the citizens and the wives of the Kuru chiefs, followed them on foot.",
  "meta": {
    "unique_id": "ANUKRAMANIKA PARVA_1_1",
    "parva_name": "ANUKRAMANIKA PARVA",
    "chapter": "1",
    "sentence": "1"
  },
  "entities": [
    {
      "start_offset": 0,
      "end_offset": 3,
      "label": "EVENT"
    },
    {
      "start_offset": 5,
      "end_offset": 8,
      "label": "EVENT"
    },
    {
      "start_offset": 9,
      "end_offset": 19,
      "label": "EVENT"
    },
    {
      "start_offset": 16,
      "end_offset": 19,
      "label": "EVENT"
    },
    {
      "start_offset": 20,
      "end_offset": 23,
      "label": "EVENT"
    },
    {
      "start_offset": 25,
      "end_offset": 33,
      "label": "EVENT"
    },
    {
      "start_offset": 30,
      "end_offset": 33,
      "label": "EVENT"
    },
    {
      

In [8]:
dsf443c

NameError: name 'dsf443c' is not defined

In [ ]:
output_df = pd.DataFrame({
    'text': df['translated_text'],
    'meta.unique_id': df['parva_name'] + '_' + df['chapter_number'].astype(str) + '_' + df['sentence_id'].astype(str),
    'meta.parva_name': df['parva_name'],
    'meta.chapter': df['chapter_number'],
    'meta.sentence': df['sentence_id']
})

with open("./mahabharata/itihasa/gen/mahabharata/mahabharata_for_doccano.jsonl", 'w', encoding='utf-8') as f:
    for _, row in output_df.iterrows():
        f.write(json.dumps({
            "text": row['text'],
            "meta": {
                k.replace('meta.', ''): v for k, v in row.items() 
                if k.startswith('meta.')
            },
            "entities": [],
            "relations": []
        }, ensure_ascii=False) + '\n')
output_df.head()

,text,meta.unique_id,meta.parva_name,meta.chapter,meta.sentence
0,Having saluted the Supreme Deity (Narayana) an...,ANUKRAMANIKA PARVA_1_1,ANUKRAMANIKA PARVA,1,1
1,One day when the great sages of hard austeriti...,ANUKRAMANIKA PARVA_1_2,ANUKRAMANIKA PARVA,1,2
2,(Thereupon) desirous of hearing his wonderful ...,ANUKRAMANIKA PARVA_1_3,ANUKRAMANIKA PARVA,1,3
3,Having been welcomed with due respect by those...,ANUKRAMANIKA PARVA_1_4,ANUKRAMANIKA PARVA,1,4
4,"After the Rishis had taken their seats, Lomaha...",ANUKRAMANIKA PARVA_1_5,ANUKRAMANIKA PARVA,1,5
